# Chargeback acceptance test

This notebook proves the correlation chain end to end:

```text
authenticated consumer -> requestId -> traceId -> model spans -> token usage
```

Each stage asserts before the next one runs. The notebook fails loudly rather
than producing a plausible number: a request with no measured usage is never
priced.

**Prerequisites** (see [SETUP.md](../SETUP.md)):

1. The instrumented agent is deployed and `main.py` is still its entry point.
2. Application Insights is connected to the Foundry project.
3. The APIM correlation policy is applied.
4. `.env` is filled in from `.env.example`.
5. `az login` has been run, and that identity can query the Application
   Insights component.

No keys are stored in this notebook. Telemetry is read with the signed-in
Azure identity.

## Setup

The kit is imported from `src/`, so a clone works without installing the
package. `pandas` is optional and only used for display.

In [ ]:
import sys
import time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))

from foundry_chargeback_kit.config import load_settings
from foundry_chargeback_kit.correlation import mint_request_id, traceparent_for
from foundry_chargeback_kit.gateway import call_agent
from foundry_chargeback_kit.showback import Rates, allocate_fixed_cost, variable_cost
from foundry_chargeback_kit.telemetry_query import (
    TelemetryClient,
    aggregate_usage,
    model_usage_spans,
    trace_ids,
)

try:
    import pandas as pd
except ImportError:
    pd = None


def show(rows, columns=None):
    """Render span rows as a table when pandas is available, otherwise as text."""
    if not rows:
        print('(no rows)')
        return None
    if pd is None:
        for row in rows:
            print({k: row.get(k) for k in (columns or row)})
        return None
    frame = pd.DataFrame(rows)
    return frame[columns] if columns else frame


settings = load_settings(ROOT / '.env')
print('project root :', ROOT)
print('apim base    :', settings.apim_base or 'MISSING')
print('agent        :', settings.agent_id or 'MISSING')
print('apim key     :', 'set' if settings.apim_subscription_key else 'MISSING - calls will 401')
print('app insights :', settings.app_insights_app_id or 'MISSING')
print('cost centre  :', settings.cost_centre)

## Stage 0 — preflight

Fail here rather than halfway through a billed request.

`APPLICATIONINSIGHTS_APP_ID` must be the component's **Application ID** from
Application Insights > API Access. It is not the resource name, the
instrumentation key, or the connection string.

In [ ]:
settings.require('apim_base', 'agent_id', 'agent_route', 'app_insights_app_id')

telemetry = TelemetryClient(settings)
probe_rows = telemetry.query('union isfuzzy=true requests, dependencies | limit 1 | project timestamp')
print('Application Insights reachable, rows returned:', len(probe_rows))

sample_id = mint_request_id()
print('sample request id :', sample_id)
print('sample traceparent:', traceparent_for(sample_id))
print('\npreflight OK')

## Stage 1 — the gateway preserves the correlation key

One request through APIM. The same lowercase 32-hex value is used for
`x-request-id` and as the `traceparent` trace ID, because Agent Server gives the
active W3C trace ID precedence over the plain header. Keeping them identical
makes the gateway ledger key, `operation_Id` and `chargeback.request.id`
directly joinable.

A pass requires APIM to echo back exactly the value that was sent.

In [ ]:
PROMPT = 'For chargeback telemetry verification, reply with the single word telemetry-ok.'

request_id = mint_request_id()
print('issuing APIM request:', request_id)

result = call_agent(settings, PROMPT, request_id, settings.cost_centre)

print('status      :', result.status, f'({result.latency_ms} ms)')
print('echoed      :', result.echoed_request_id)
print('response id :', result.response_id)
print('cost centre :', result.cost_centre)

assert result.ok, f'APIM returned HTTP {result.status}: {str(result.body)[:1000]}'
assert result.correlation_echoed, (
    f'APIM echoed x-request-id={result.echoed_request_id!r}, expected {request_id!r}'
)
assert result.response_id, 'No Foundry response_id returned; telemetry cannot be diagnosed.'
print('\nPASS: request ID forwarded and echoed')

## Stage 2 — the hosted agent stamped the key onto its spans

`ChargebackSpanProcessor` reads `x_request_id` from OpenTelemetry baggage and
writes it to `chargeback.request.id` on every agent, tool and model span.

Ingestion is asynchronous, so this polls. An empty result after the timeout
usually means one of:

- `main.py` is no longer the deployed entry point, so the callback never ran;
- Application Insights is not connected to the Foundry project;
- the query targets a different Application Insights component.

In [ ]:
print(f'polling for up to {settings.telemetry_timeout_seconds}s ...')
started = time.perf_counter()
spans = telemetry.await_model_usage(request_id)
print(f'returned after {time.perf_counter() - started:.0f}s with {len(spans)} spans')

if not spans:
    print('\nNo spans carried the request ID. Falling back to the response-ID diagnostic.')
    diagnostic = telemetry.spans_for_response(result.response_id)
    print('spans found by response id:', len(diagnostic))
    if diagnostic:
        observed = sorted({str(s.get('requestId') or '') for s in diagnostic} - {''})
        print('request IDs on those spans:', observed or '(none - span processor not active)')
    raise AssertionError('No spans carried chargeback.request.id.')

show(spans, ['timestamp', 'telemetryTable', 'name', 'operationName',
             'inputTokens', 'outputTokens', 'requestId'])

## Stage 3 — the spans belong to one distributed operation

A shared `chargeback.request.id` alone is not proof: it is an attribute the
application writes. The trace ID is independent evidence produced by the
OpenTelemetry context, so both must agree.

The tree below should show the hosted invocation as root, with orchestrator
model calls, tool calls, and any delegated specialist calls beneath it —
including calls made concurrently.

In [ ]:
observed_traces = trace_ids(spans)
print('operation_Id:', ', '.join(sorted(observed_traces)))

assert len(observed_traces) == 1, (
    f'Spans span {len(observed_traces)} traces; execution lineage is not provable.'
)

by_id = {span['id']: span for span in spans}
children = {}
for span in spans:
    children.setdefault(span.get('operation_ParentId'), []).append(span)
roots = [span for span in spans if span.get('operation_ParentId') not in by_id]


def walk(span, depth=0):
    kind = span.get('operationName') or span.get('telemetryTable')
    tokens = ''
    if span['inputTokens'] or span['outputTokens']:
        tokens = '  in={} out={}'.format(span['inputTokens'], span['outputTokens'])
    print('{}- {} [{}]{}'.format('  ' * depth, span['name'], kind, tokens))
    for child in children.get(span['id'], []):
        walk(child, depth + 1)


print()
for root in roots:
    walk(root)

print('\nPASS: one trace, correlated lineage')

## Stage 4 — measured usage

Only deduplicated `chat` spans are counted. The `invoke_agent` span repeats the
usage attributes of its child `chat` span, so counting both would double charge.

Missing `gen_ai.usage.*` attributes are **missing data**, not zero-cost usage.
If they are absent, fix telemetry at the model-call producer instead of
treating the gap as free.

In [ ]:
priced_spans = model_usage_spans(spans)
usage = aggregate_usage(spans)

print(f'spans returned    : {len(spans)}')
print(f'priced model spans: {len(priced_spans)}')
print(f'models            : {", ".join(usage.models) or "unknown"}')
print(f'input tokens      : {usage.input_tokens}')
print(f'  of which cached : {usage.cached_tokens}')
print(f'output tokens     : {usage.output_tokens}')
print(f'  of which reason.: {usage.reasoning_tokens}')

assert usage.measured, 'No model span reported measured token usage; showback would be fabricated.'

show(priced_spans, ['timestamp', 'name', 'responseModel',
                    'inputTokens', 'cachedTokens', 'outputTokens', 'reasoningTokens'])

## Stage 5 — price it

`gen_ai.usage.input_tokens` includes cached tokens, which bill at a discounted
rate, so they are subtracted before the standard input rate is applied.

The rates below are indicative placeholders. Replace `Rates.indicative()` with
your contracted price list, versioned by provider, deployment, region, meter and
effective date.

In [ ]:
rates = Rates.indicative()
cost = variable_cost(usage, rates)

print(f'estimated variable cost: {cost:.8f} {rates.currency}')
print('(indicative rates - not an invoiced amount)')

# The row a gateway ledger would carry. No keys, prompts or completions.
ledger_row = {
    'requestId': request_id,
    'operationId': sorted(observed_traces)[0],
    'responseId': result.response_id,
    'costCentre': result.cost_centre,
    'httpStatus': result.status,
    'durationMs': result.latency_ms,
    'inputTokens': usage.input_tokens,
    'cachedTokens': usage.cached_tokens,
    'outputTokens': usage.output_tokens,
    'reasoningTokens': usage.reasoning_tokens,
    'models': list(usage.models),
    'estimatedVariableCost': round(cost, 8),
    'currency': rates.currency,
}
ledger_row

## Stage 6 — showback across cost centres (optional)

Sends several requests attributed to different cost centres, then apportions
the always-on hosted compute by request share.

This issues real billable calls and waits for ingestion on each one, so it is
off by default. In production the cost centre comes from validated claims at
APIM, never from the caller header used here.

In [ ]:
RUN_SHOWBACK = True
COST_CENTRES = ['CC-1000', 'CC-2000']
REQUESTS_PER_CENTRE = 1
FIXED_COST_PER_HOUR = 0.60  # always-on hosted-agent compute
PERIOD_HOURS = 1

if not RUN_SHOWBACK:
    print('Skipped. Set RUN_SHOWBACK = True to issue billable requests.')
else:
    measured, unmeasured = [], []
    for centre in COST_CENTRES:
        for _ in range(REQUESTS_PER_CENTRE):
            rid = mint_request_id()
            call = call_agent(settings, PROMPT, rid, centre)
            if not (call.ok and call.correlation_echoed):
                unmeasured.append({'requestId': rid, 'costCentre': centre, 'reason': 'gateway'})
                continue
            centre_usage = aggregate_usage(telemetry.await_model_usage(rid))
            if not centre_usage.measured:
                unmeasured.append({'requestId': rid, 'costCentre': centre, 'reason': 'no usage'})
                continue
            measured.append({
                'requestId': rid,
                'costCentre': centre,
                'inputTokens': centre_usage.input_tokens,
                'cachedTokens': centre_usage.cached_tokens,
                'outputTokens': centre_usage.output_tokens,
                'variableCost': variable_cost(centre_usage, rates),
            })
            print(f'{centre} {rid} priced')

    assert measured, 'No request had measured usage; showback would be fabricated.'

    counts = {}
    for row in measured:
        counts[row['costCentre']] = counts.get(row['costCentre'], 0) + 1
    allocated = allocate_fixed_cost(counts, FIXED_COST_PER_HOUR, PERIOD_HOURS)

    showback = []
    for centre, count in counts.items():
        rows = [r for r in measured if r['costCentre'] == centre]
        variable = sum(r['variableCost'] for r in rows)
        showback.append({
            'costCentre': centre,
            'requests': count,
            'inputTokens': sum(r['inputTokens'] for r in rows),
            'outputTokens': sum(r['outputTokens'] for r in rows),
            'variableCost': round(variable, 8),
            'fixedAllocated': round(allocated[centre], 8),
            'totalCost': round(variable + allocated[centre], 8),
        })

    print(f'\npriced requests  : {len(measured)}')
    print(f'excluded requests: {len(unmeasured)} (excluded, never assumed free)')
    display(show(showback))

## What a pass proves

- APIM preserved and echoed the business request ID, including on failure if
  the `on-error` handler is in place.
- The hosted agent stamped that ID onto every span it produced, so usage
  generated *inside* the container is attributable even though it never
  traversed the gateway.
- The trace ID independently confirms those spans are one distributed
  operation.
- Token usage was measured on deduplicated model spans, not inferred.

## What it does not prove

- That the figure equals an invoice. Reconcile against Azure Cost Management by
  billing period, resource, deployment, meter and region before presenting any
  number as chargeback.
- That fixed costs are traced. Hosted compute, APIM capacity and telemetry
  ingestion are apportioned by rule and must be labelled as allocation.
- That per-user attribution is possible. When the authenticated caller is a
  shared service principal, cost-centre scope is the honest ceiling.

For a non-interactive equivalent, run `python -m foundry_chargeback_kit.cli e2e --json`.